In [4]:
import pandas as pd
import requests
import nltk
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
from tqdm import tqdm
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from functools import lru_cache

# Khởi tạo thanh tiến trình cho Pandas
tqdm.pandas()

# Tải gói dữ liệu NLTK
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('omw-1.4', quiet=True)

# 1. HÀM WORDNET với CACHE (tránh xử lý lại văn bản giống nhau)
@lru_cache(maxsize=50000)
def enrich_with_wordnet(text):
    enriched_words = []
    words = word_tokenize(str(text))
    
    for word in words:
        synsets = wordnet.synsets(word)
        if synsets:
            primary_synset = synsets[0]
            # Lấy 1 từ đồng nghĩa
            for lemma in primary_synset.lemmas()[:1]: 
                enriched_words.append(lemma.name().replace('_', ' '))
            # Lấy 1 từ bao nghĩa (khái niệm rộng hơn)
            for hypernym in primary_synset.hypernyms()[:1]: 
                enriched_words.append(hypernym.lemmas()[0].name().replace('_', ' '))
                
    return ' '.join(set(enriched_words))

# 2. HÀM DBPEDIA với THREADING (nhanh hơn nhiều, tương thích notebook)
@lru_cache(maxsize=50000)
def link_to_dbpedia(text):
    """Gọi DBpedia API với cache"""
    url = "https://api.dbpedia-spotlight.org/en/annotate"
    params = {"text": str(text), "confidence": 0.5}
    headers = {"Accept": "application/json"}
    
    try:
        response = requests.get(url, params=params, headers=headers, timeout=10)
        if response.status == 200:
            data = response.json()
            uris = []
            if 'Resources' in data:
                for resource in data['Resources']:
                    uri_tail = resource['@URI'].split('/')[-1]
                    uris.append(uri_tail)
            time.sleep(0.05)  # Delay rất nhỏ
            return ' '.join(uris)
    except Exception:
        pass
    
    time.sleep(0.05)
    return ""

def process_dbpedia_parallel(texts, max_workers=20):
    """
    Xử lý DBpedia với ThreadPoolExecutor - nhanh và tương thích notebook
    max_workers=20: Gọi 20 API đồng thời (nhanh gấp 20 lần)
    """
    results = [None] * len(texts)
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit tất cả tasks
        future_to_index = {executor.submit(link_to_dbpedia, text): i 
                          for i, text in enumerate(texts)}
        
        # Thu thập kết quả với progress bar
        for future in tqdm(as_completed(future_to_index), 
                          total=len(texts), 
                          desc="DBpedia"):
            index = future_to_index[future]
            results[index] = future.result()
    
    return results

# ==========================================
# THỰC THI TRÊN DỮ LIỆU
# ==========================================

print("1. Đang đọc dữ liệu df_clean...")
df_clean = pd.read_csv("../data/processed/df_clean.csv")
print(f"   Tổng số dòng: {len(df_clean):,}")

# Kiểm tra văn bản trùng lặp
unique_texts = df_clean['text'].nunique()
print(f"   Văn bản unique: {unique_texts:,} ({unique_texts/len(df_clean)*100:.1f}%)")

print("\n2. Đang chạy WordNet với Threading (8 workers)...")
start = time.time()
with ThreadPoolExecutor(max_workers=8) as executor:
    df_clean['wordnet_features'] = list(tqdm(
        executor.map(enrich_with_wordnet, df_clean['text']), 
        total=len(df_clean),
        desc="WordNet"
    ))
wordnet_time = time.time() - start
print(f"   ✓ Hoàn thành trong {wordnet_time:.1f}s")

print("\n3. Đang chạy DBpedia Spotlight với Threading (20 workers đồng thời)...")
print("   Ước tính: ~40-60 phút cho 120,000 dòng (nhanh gấp 20 lần)")
start = time.time()
df_clean['dbpedia_features'] = process_dbpedia_parallel(df_clean['text'].tolist(), max_workers=20)
dbpedia_time = time.time() - start
print(f"   ✓ Hoàn thành trong {dbpedia_time/60:.1f} phút")

# Lưu lại file
df_clean.to_csv("../data/processed/df_enriched.csv", index=False)
print(f"\n✓ Đã lưu file df_enriched.csv")
print(f"✓ Tổng thời gian: {(wordnet_time + dbpedia_time)/60:.1f} phút")

# Xem thử kết quả
print("\n" + "="*60)
print("SAMPLE KẾT QUẢ:")
print(df_clean[['text', 'wordnet_features', 'dbpedia_features']].head(3))

1. Đang đọc dữ liệu df_clean...
   Tổng số dòng: 120,000
   Văn bản unique: 120,000 (100.0%)

2. Đang chạy WordNet với Threading (8 workers)...


WordNet: 100%|██████████| 120000/120000 [01:15<00:00, 1593.74it/s] 


   ✓ Hoàn thành trong 88.4s

3. Đang chạy DBpedia Spotlight với Threading (20 workers đồng thời)...
   Ước tính: ~40-60 phút cho 120,000 dòng (nhanh gấp 20 lần)


DBpedia: 100%|██████████| 120000/120000 [1:14:51<00:00, 26.72it/s]


   ✓ Hoàn thành trong 74.9 phút

✓ Đã lưu file df_enriched.csv
✓ Tổng thời gian: 76.4 phút

SAMPLE KẾT QUẢ:
                                                text  \
0  Wall St. Bears Claw Back Into the Black (Reute...   
1  Carlyle Looks Toward Commercial Aerospace (Reu...   
2  Oil and Economy Cloud Stocks' Outlook (Reuters...   

                                    wordnet_features dbpedia_features  
0  carnivore body part black again street back gr...                   
1  defense commercial Carlyle linear unit gamble ...                   
2  stagnation attitude activity time period exten...                   


note:
Máy sẽ tạo ra thêm 2 cột mới trong df_clean của bạn.

Cột wordnet_features sẽ chứa các từ vựng mở rộng (ví dụ văn bản gốc có chữ "dog", cột này sẽ thêm chữ "animal").

Cột dbpedia_features sẽ chứa các URI (ví dụ văn bản gốc có chữ "Wall St.", cột này sẽ trích xuất ra "Wall_Street" chuẩn hóa từ Wikipedia)